# ⚡ Tự Động Hóa Backtest 240 Ngày Siêu Tốc Trên Google Colab Pro (High-RAM + GCS / Cloudflare R2)
### Phiên Bản: **Khung Đánh Giá Mở Rộng 240 Ngày** + Smart Caching Layer trên **Google Cloud Storage (GCS)** hoặc **Cloudflare R2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tminhnhat/NostalgiaForInfinity/blob/main/notebooks/Run_NFIX8_Backtest_Colab_GCS_HighRAM.ipynb)

---
### ☁️ CẤU TRÚC THƯ MỤC CACHE TRÊN CLOUD STORAGE (`<BUCKET>/<PREFIX>/`):
1. **`binance_candles_240d.tar.gz`** (hoặc `binance_candles_185d.tar.gz`): Dữ liệu nến lưu trên GCS hoặc Cloudflare R2.
2. **`talib_c.tar.gz` & `wheels_cache.tar.gz`** (Thư viện Linux x86_64): Colab tự động biên dịch và lưu lên Cloud Storage ở lần chạy đầu tiên.
3. **`reports/`**: Tự động đồng bộ báo cáo và file kết quả backtest lên Cloud Storage.

---
### 🚀 BẢNG SO SÁNH TỐC ĐỘ TRƯỚC VÀ SAU KHI TỐI ƯU:
| Hạng mục | Quy trình cũ (chậm) | Quy trình tối ưu với Cloud Smart Cache | Cải thiện |
| :--- | :--- | :--- | :--- |
| **Cài TA-Lib C & Pip** | ~3 - 5 phút (compile C & tải 50 gói) | **~5 - 10 giây** (nạp từ Cloud cache) | **Nhanh gấp 25 lần** |
| **Nạp dữ liệu nến 240 ngày** | ~15 - 25 phút (hàng vạn request) | **~3 - 15 giây** (tải bundle từ GCS/R2 hoặc sync bù) | **Nhanh gấp 100 lần** ⚡ |
| **Chạy Backtest 240 ngày** | ~38 - 45 phút trên i5 cũ | **~4 - 5 phút** (8 vCPUs High-RAM) | **Nhanh gấp 8 lần** |
| **Tổng thời gian từ A - Z** | **20 - 35 phút** | **~4.5 - 5.5 phút** | **Tiết kiệm 85% thời gian** |

> 💡 **CÁCH SỬ DỤNG:**
> 1. Vào **Runtime ➔ Change runtime type** ➔ Chọn **None (CPU)** và **High-RAM**.
> 2. Ở **Bước 1**, chọn `STORAGE_PROVIDER` là **`GCS`** hoặc **`R2`** (Cloudflare R2) và điền `STORAGE_BUCKET`.
>    - **Nếu dùng GCS**: có thể đăng nhập qua popup Google, hoặc đặt Colab Secret `GCS_SA_KEY_JSON`.
>    - **Nếu dùng Cloudflare R2**: điền `R2_ACCOUNT_ID`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY` trực tiếp hoặc lưu trong Colab Secrets cùng tên.
> 3. Bấm **Ctrl + F9** (hoặc **Runtime ➔ Run all**).

In [ ]:
# ==============================================================================
# BƯỚC 1: KIỂM TRA PHẦN CỨNG & KẾT NỐI CLOUD STORAGE (GCS HOẶC CLOUDFLARE R2)
# ==============================================================================
import os
import shutil
import subprocess
import psutil
from pathlib import Path

#@title ⚙️ Cấu hình Cloud Storage (GCS / Cloudflare R2)
STORAGE_PROVIDER = "GCS"  #@param ["GCS", "R2"]
STORAGE_BUCKET = "botx8-cache"  #@param {type:"string"}
STORAGE_PREFIX = "botx8_cache"  #@param {type:"string"}
GCS_SA_KEY_PATH = ""  #@param {type:"string"}
R2_ACCOUNT_ID = ""  #@param {type:"string"}
R2_ACCESS_KEY_ID = ""  #@param {type:"string"}
R2_SECRET_ACCESS_KEY = ""  #@param {type:"string"}

vcpus = os.cpu_count()
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"🖥️  Số luồng CPU (vCPUs): {vcpus} cores")
print(f"🧠  Dung lượng RAM:       {ram_gb:.1f} GB")
print(f"☁️  Storage Provider:     {STORAGE_PROVIDER}")

LOCAL_CACHE_DIR = Path('/content/cloud_cache')
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

bucket_clean = STORAGE_BUCKET.strip().removeprefix('gs://').removeprefix('s3://').removeprefix('r2://').strip('/')
prefix_clean = STORAGE_PREFIX.strip().strip('/')
USE_GCLOUD_STORAGE = shutil.which('gcloud') is not None
R2_CLIENT = None
STORAGE_ACTIVE = False

if STORAGE_PROVIDER == 'R2':
    STORAGE_BASE_URI = f"r2://{bucket_clean}/{prefix_clean}" if prefix_clean else f"r2://{bucket_clean}"
else:
    STORAGE_BASE_URI = f"gs://{bucket_clean}/{prefix_clean}" if prefix_clean else f"gs://{bucket_clean}"

def _get_secret(name: str, fallback: str = "") -> str:
    val = (fallback or "").strip()
    if val:
        return val
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return os.environ.get(name, "").strip()

def _r2_key(rel_path: str) -> str:
    rel = rel_path.lstrip('/')
    return f"{prefix_clean}/{rel}" if prefix_clean else rel

def storage_uri(rel_path: str) -> str:
    rel = rel_path.lstrip('/')
    return f"{STORAGE_BASE_URI}/{rel}"

def storage_exists(rel_path: str) -> bool:
    if not STORAGE_ACTIVE:
        return False
    if STORAGE_PROVIDER == 'R2':
        try:
            R2_CLIENT.head_object(Bucket=bucket_clean, Key=_r2_key(rel_path))
            return True
        except Exception:
            return False
    uri = storage_uri(rel_path)
    cmd = ['gcloud', 'storage', 'ls', uri] if USE_GCLOUD_STORAGE else ['gsutil', '-q', 'stat', uri]
    return subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0

def storage_download(rel_path: str, local_path: Path) -> bool:
    if not STORAGE_ACTIVE:
        return False
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    uri = storage_uri(rel_path)
    if STORAGE_PROVIDER == 'R2':
        try:
            R2_CLIENT.download_file(bucket_clean, _r2_key(rel_path), str(local_path))
            return True
        except Exception as e:
            print(f"⚠️ Lỗi tải {uri}: {e}")
            return False
    cmd = ['gcloud', 'storage', 'cp', uri, str(local_path)] if USE_GCLOUD_STORAGE else ['gsutil', '-q', 'cp', uri, str(local_path)]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"⚠️ Lỗi tải {uri}: {res.stderr.strip()}")
        return False
    return True

def storage_upload(local_path: Path, rel_path: str) -> bool:
    if not STORAGE_ACTIVE or not Path(local_path).exists():
        return False
    uri = storage_uri(rel_path)
    if STORAGE_PROVIDER == 'R2':
        try:
            R2_CLIENT.upload_file(str(local_path), bucket_clean, _r2_key(rel_path))
            return True
        except Exception as e:
            print(f"⚠️ Lỗi upload lên {uri}: {e}")
            return False
    cmd = ['gcloud', 'storage', 'cp', str(local_path), uri] if USE_GCLOUD_STORAGE else ['gsutil', '-q', 'cp', str(local_path), uri]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"⚠️ Lỗi upload lên {uri}: {res.stderr.strip()}")
        return False
    return True

def storage_upload_dir(local_dir: Path, rel_dir: str) -> bool:
    local_dir = Path(local_dir)
    if not STORAGE_ACTIVE or not local_dir.exists():
        return False
    if STORAGE_PROVIDER == 'R2':
        ok_any = False
        for p in local_dir.rglob('*'):
            if p.is_file():
                sub = p.relative_to(local_dir).as_posix()
                if storage_upload(p, f"{rel_dir.strip('/')}/{sub}"):
                    ok_any = True
        return ok_any
    uri = storage_uri(rel_dir).rstrip('/') + '/'
    src = str(local_dir).rstrip('/') + '/*'
    cmd = f'gcloud storage cp -r {src} {uri}' if USE_GCLOUD_STORAGE else f'gsutil -m cp -r {src} {uri}'
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"⚠️ Lỗi đồng bộ thư mục lên {uri}: {res.stderr.strip()}")
        return False
    return True

if not bucket_clean:
    print("⚠️ Chưa cấu hình STORAGE_BUCKET. Sẽ sử dụng local cache tạm thời.")
elif STORAGE_PROVIDER == 'R2':
    try:
        acct_id = _get_secret('R2_ACCOUNT_ID', R2_ACCOUNT_ID)
        access_key = _get_secret('R2_ACCESS_KEY_ID', R2_ACCESS_KEY_ID)
        secret_key = _get_secret('R2_SECRET_ACCESS_KEY', R2_SECRET_ACCESS_KEY)
        assert acct_id and access_key and secret_key, 'Thiếu R2_ACCOUNT_ID / R2_ACCESS_KEY_ID / R2_SECRET_ACCESS_KEY'
        subprocess.run(['pip', 'install', '-q', 'boto3'], check=True)
        import boto3
        from botocore.config import Config
        R2_CLIENT = boto3.client(
            's3',
            endpoint_url=f'https://{acct_id}.r2.cloudflarestorage.com',
            aws_access_key_id=access_key,
            aws_secret_access_key=secret_key,
            region_name='auto',
            config=Config(signature_version='s3v4')
        )
        R2_CLIENT.head_bucket(Bucket=bucket_clean)
        STORAGE_ACTIVE = True
        print(f"✅ Đã kết nối Cloudflare R2! Cache URI: {STORAGE_BASE_URI}")
    except Exception as e:
        STORAGE_ACTIVE = False
        print(f"⚠️ Không kết nối được Cloudflare R2 ({e}). Sẽ sử dụng local cache tạm thời.")
else:
    try:
        sa_key_file = None
        if GCS_SA_KEY_PATH and Path(GCS_SA_KEY_PATH).exists():
            sa_key_file = GCS_SA_KEY_PATH
        else:
            sa_json = _get_secret('GCS_SA_KEY_JSON')
            if sa_json:
                sa_tmp = Path('/content/.gcs_sa_key.json')
                sa_tmp.write_text(sa_json, encoding='utf-8')
                sa_key_file = str(sa_tmp)

        if sa_key_file:
            subprocess.run(['gcloud', 'auth', 'activate-service-account', f'--key-file={sa_key_file}'], check=True)
            print("🔑 Đã xác thực GCS bằng Service Account.")
        else:
            from google.colab import auth
            auth.authenticate_user()
            print("🔑 Đã xác thực GCS qua tài khoản Google Colab.")

        check_cmd = ['gcloud', 'storage', 'ls', f'gs://{bucket_clean}'] if USE_GCLOUD_STORAGE else ['gsutil', 'ls', f'gs://{bucket_clean}']
        subprocess.run(check_cmd, check=True, stdout=subprocess.DEVNULL)
        STORAGE_ACTIVE = True
        print(f"✅ Đã kết nối Google Cloud Storage! Cache URI: {STORAGE_BASE_URI}")
    except Exception as e:
        STORAGE_ACTIVE = False
        print(f"⚠️ Không kết nối được GCS ({e}). Sẽ sử dụng local cache tạm thời.")

In [ ]:
# ==============================================================================
# BƯỚC 2: CÀI ĐẶT TA-LIB & THƯ VIỆN BẰNG CLOUD CACHE (GIẢM TỪ 4 PHÚT XUỐNG 10 GIÂY)
# ==============================================================================
import os
import subprocess
import time
from pathlib import Path

# Nâng cấp cffi trước để giải quyết triệt để lỗi xung đột với curl-cffi của Freqtrade
print('⏳ Đang chuẩn bị môi trường và nâng cấp cffi...')
!pip install -q --upgrade "cffi>=2.1.1" setuptools wheel

TALIB_C_CACHE = LOCAL_CACHE_DIR / 'talib_c.tar.gz'
WHEELS_CACHE = LOCAL_CACHE_DIR / 'wheels_cache.tar.gz'
PIP_WHEELS_DIR = Path('/root/.cache/pip/wheels')
PIP_WHEELS_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()
has_cache = False

if STORAGE_ACTIVE and storage_exists('talib_c.tar.gz') and storage_exists('wheels_cache.tar.gz'):
    print(f'⚡ TÌM THẤY CACHE THƯ VIỆN LINUX TRÊN {STORAGE_PROVIDER}!')
    if storage_download('talib_c.tar.gz', TALIB_C_CACHE) and storage_download('wheels_cache.tar.gz', WHEELS_CACHE):
        has_cache = True
elif TALIB_C_CACHE.exists() and WHEELS_CACHE.exists():
    has_cache = True

if has_cache:
    print('⏳ Đang nạp TA-Lib C và giải nén Python Wheels...')
    !tar -xzf {TALIB_C_CACHE} -C / 2>/dev/null
    !ldconfig 2>/dev/null
    !tar -xzf {WHEELS_CACHE} -C /root/.cache/pip/ 2>/dev/null
    !pip install -q --find-links=/root/.cache/pip/wheels freqtrade ta-lib lightgbm scikit-learn joblib python-rapidjson pyarrow fastparquet
    print('✅ Đã nạp môi trường từ cache!')

# Tự động kiểm tra an toàn: Nếu TA-Lib C chưa có (cache hỏng), tự động biên dịch lại ngay
if not os.path.exists('/usr/lib/libta_lib.so.0'):
    print('⏳ Đang biên dịch TA-Lib C library...')
    install_talib_script = '''
    wget -q http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
    tar -xzf ta-lib-0.4.0-src.tar.gz
    cd ta-lib/
    ./configure --prefix=/usr > /dev/null 2>&1
    make -j$(nproc) > /dev/null 2>&1
    sudo make install > /dev/null 2>&1
    sudo ldconfig > /dev/null 2>&1
    cd ..
    rm -rf ta-lib ta-lib-0.4.0-src.tar.gz
    '''
    subprocess.run(install_talib_script, shell=True, check=True, executable='/bin/bash')
    !pip install -q ta-lib freqtrade lightgbm scikit-learn joblib python-rapidjson pyarrow fastparquet

# Kiểm tra an toàn: Đảm bảo import thành công các thư viện cốt lõi
try:
    import talib
    import freqtrade
    import lightgbm
except ImportError:
    print('⚠️ Phát hiện thiếu package, đang tự động cài bổ sung từ PyPI...')
    !pip install -q ta-lib freqtrade lightgbm scikit-learn joblib python-rapidjson pyarrow fastparquet

# Tự động lưu cache mới chuẩn lên Cloud Storage nếu chưa có
if STORAGE_ACTIVE and not has_cache:
    print(f'💾 Đang tự động đóng gói & upload cache thư viện Linux lên {STORAGE_PROVIDER}...')
    try:
        !tar -czf {TALIB_C_CACHE} /usr/lib/libta_lib* /usr/include/ta-lib 2>/dev/null
        !tar -czf {WHEELS_CACHE} -C /root/.cache/pip wheels 2>/dev/null
        ok1 = storage_upload(TALIB_C_CACHE, 'talib_c.tar.gz')
        ok2 = storage_upload(WHEELS_CACHE, 'wheels_cache.tar.gz')
        if ok1 and ok2:
            print(f'✅ Đã lưu cache thư viện vào: {STORAGE_BASE_URI}')
    except Exception as e:
        print(f'⚠️ Lưu cache thư viện: {e}')

# Xác minh cuối cùng
import talib
import freqtrade
import lightgbm
print(f'\n🎉 MÔI TRƯỜNG HOÀN TOÀN SẴN SÀNG TRONG {time.time()-t0:.1f}s!')
print(f'   - Freqtrade: {freqtrade.__version__}')
print(f'   - TA-Lib:    {talib.__version__}')
print(f'   - LightGBM:  {lightgbm.__version__}')


In [ ]:
# ==============================================================================
# BƯỚC 3: CLONE HOẶC CẬP NHẬT MÃ NGUỒN BOT MỚI NHẤT TỪ GITHUB (MAIN)
# ==============================================================================
import os
from pathlib import Path

REPO_URL = 'https://github.com/tminhnhat/NostalgiaForInfinity.git'
REPO_DIR = '/content/NostalgiaForInfinity'

if not os.path.exists(REPO_DIR):
    print(f'📥 Đang clone repository từ: {REPO_URL}...')
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print('🔄 Đang đồng bộ bắt buộc mã nguồn mới nhất từ GitHub main...')
    %cd {REPO_DIR}
    !git fetch origin main
    !git reset --hard origin/main

%cd {REPO_DIR}

# Xác nhận các file chiến lược và mô hình
assert Path('NostalgiaForInfinityX8_Combined_AI.py').exists(), 'Thiếu file chiến lược!'
assert Path('configs/backtest_config.json').exists(), 'Thiếu file config backtest!'
assert Path('user_data/models/nfix8_short_ai_v3.pkl').exists(), 'Thiếu file mô hình AI!'
print('✅ Chiến lược & Mô hình AI Gatekeeper v3 đã sẵn sàng trên commit mới nhất!')


In [ ]:
# ==============================================================================
# BƯỚC 4: NẠP DỮ LIỆU NẾN 240 NGÀY SIÊU TỐC TỪ CLOUD STORAGE & TỰ ĐỘNG SYNC
# ==============================================================================
import time
from pathlib import Path

CANDLE_BUNDLE_240D = LOCAL_CACHE_DIR / 'binance_candles_240d.tar.gz'
CANDLE_BUNDLE_185D = LOCAL_CACHE_DIR / 'binance_candles_185d.tar.gz'
DEST_DIR = Path('/content/NostalgiaForInfinity/user_data/data/binance')
DEST_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()
if STORAGE_ACTIVE and storage_exists('binance_candles_240d.tar.gz'):
    print(f'⚡ TÌM THẤY GÓI DỮ LIỆU NẾN 240 NGÀY TRÊN {STORAGE_PROVIDER}: {storage_uri("binance_candles_240d.tar.gz")}')
    if storage_download('binance_candles_240d.tar.gz', CANDLE_BUNDLE_240D):
        !tar -xzf {CANDLE_BUNDLE_240D} -C {DEST_DIR}
        CANDLE_BUNDLE_240D.unlink(missing_ok=True)
        print(f'🎉 Đã nạp toàn bộ 240 ngày nến từ {STORAGE_PROVIDER} thành công chỉ trong {time.time()-t0:.1f}s!')
elif STORAGE_ACTIVE and storage_exists('binance_candles_185d.tar.gz'):
    print(f'⚡ TÌM THẤY GÓI NẾN 185 NGÀY TRÊN {STORAGE_PROVIDER}: {storage_uri("binance_candles_185d.tar.gz")}')
    print(f'⏳ Đang tải nhanh 185 ngày từ {STORAGE_PROVIDER} và tải bổ sung ~55 ngày còn lại để đủ 240 ngày...')
    if storage_download('binance_candles_185d.tar.gz', CANDLE_BUNDLE_185D):
        !tar -xzf {CANDLE_BUNDLE_185D} -C {DEST_DIR}
        CANDLE_BUNDLE_185D.unlink(missing_ok=True)
    !python scripts/download_candles.py \
        --config configs/backtest_config.json \
        --days 240 \
        --output-dir user_data/data/binance \
        --workers 24 \
        --skip-existing
    print(f'💾 Đang tự động đóng gói & lưu gói nến 240 ngày hoàn chỉnh lên {STORAGE_PROVIDER}...')
    !tar -czf {CANDLE_BUNDLE_240D} -C user_data/data/binance futures 2>/dev/null
    if storage_upload(CANDLE_BUNDLE_240D, 'binance_candles_240d.tar.gz'):
        print(f'✅ Đã nâng cấp gói nến 240 ngày tại: {storage_uri("binance_candles_240d.tar.gz")}')
    CANDLE_BUNDLE_240D.unlink(missing_ok=True)
else:
    print(f'📥 Chưa có gói cache trên {STORAGE_PROVIDER}. Đang tải đầy đủ dữ liệu 240 ngày từ Binance Vision...')
    !python scripts/download_candles.py \
        --config configs/backtest_config.json \
        --days 240 \
        --output-dir user_data/data/binance \
        --workers 24 \
        --skip-existing
    if STORAGE_ACTIVE:
        print(f'💾 Đang tự động lưu gói nến 240 ngày lên {STORAGE_PROVIDER} cho các lần sau...')
        !tar -czf {CANDLE_BUNDLE_240D} -C user_data/data/binance futures 2>/dev/null
        if storage_upload(CANDLE_BUNDLE_240D, 'binance_candles_240d.tar.gz'):
            print(f'✅ Đã lưu gói nến 240 ngày tại: {storage_uri("binance_candles_240d.tar.gz")}')
        CANDLE_BUNDLE_240D.unlink(missing_ok=True)

# Áp dụng bản vá Offline Markets để tránh bị sàn chặn IP
!python scripts/patch_offline_markets.py
feather_count = len(list(Path('user_data/data/binance/futures').glob('*.feather')))
print(f'📊 Tổng cộng: {feather_count} file nến đã sẵn sàng trong thư mục futures!')


In [ ]:
# ==============================================================================
# BƯỚC 5: CHẠY BACKTEST 240 NGÀY SIÊU TỐC TRÊN 8 vCPUs (~4 - 5 PHÚT)
# ==============================================================================
import time

start_time = time.time()
print('🚀 BẮT ĐẦU CHẠY BACKTEST 240 NGÀY TRÊN 8 vCPUs (HIGH-RAM)...')

!python scripts/run_eval_backtest.py \
    --strategy NostalgiaForInfinityX8_Combined_AI \
    --days 240 \
    --run-label "Colab_Pro_Cloud_240d_Eval"

elapsed = time.time() - start_time
print(f'\n⏱️ TỔNG THỜI GIAN THỰC THI: {elapsed / 60:.2f} phút ({elapsed:.1f} giây)!')


In [ ]:
# ==============================================================================
# BƯỚC 6: HIỂN THỊ BÁO CÁO HIỆU SUẤT TRỰC QUAN (MARKDOWN FORMATTED)
# ==============================================================================
from IPython.display import display, Markdown
from pathlib import Path

report_file = Path("reports/latest_report.md")
if report_file.exists():
    with open(report_file, "r", encoding="utf-8") as f:
        content = f.read()
    display(Markdown(content))
else:
    print("⚠️ Không tìm thấy file reports/latest_report.md!")

In [ ]:
# ==============================================================================
# BƯỚC 7: LƯU BÁO CÁO LÊN CLOUD STORAGE & TỰ ĐỘNG GỬI QUA TELEGRAM
# ==============================================================================
import time
from pathlib import Path

UPLOAD_COMPLETED = False

# 1. Lưu vĩnh viễn báo cáo và file kết quả lên GCS / Cloudflare R2
if STORAGE_ACTIVE:
    t_up = time.time()
    print(f'☁️ Đang upload báo cáo & kết quả backtest lên {STORAGE_PROVIDER}...')
    ok_rep = storage_upload_dir(Path('reports'), 'reports')
    ok_bt = storage_upload_dir(Path('user_data/backtest_results'), 'reports/backtest_results')
    verified = storage_exists('reports/latest_report.md') and storage_exists('reports/latest_trades.csv')
    UPLOAD_COMPLETED = bool((ok_rep or ok_bt) and verified)
    if UPLOAD_COMPLETED:
        print(f'💾 Đã lưu & xác minh báo cáo lên {STORAGE_PROVIDER} trong {time.time()-t_up:.1f}s: {storage_uri("reports")}')
    else:
        print(f'⚠️ Upload lên {STORAGE_PROVIDER} chưa hoàn tất hoặc chưa xác minh được (ok_rep={ok_rep}, ok_bt={ok_bt}, verified={verified})!')
else:
    UPLOAD_COMPLETED = True

# 2. Bắn thông báo tóm tắt và đính kèm file latest_report.md về Telegram
!python scripts/notify_telegram.py \
    --report reports/latest_report.md \
    --context-title '⚡ BÁO CÁO BACKTEST 240 NGÀY COLAB PRO (CLOUD STORAGE + HIGH-RAM)'

# Chờ 5s để đảm bảo toàn bộ I/O mạng hoàn tất trước bước ngắt runtime
time.sleep(5)

In [ ]:
# ==============================================================================
# BƯỚC 8: TỰ ĐỘNG NGẮT KẾT NỐI RUNTIME ĐỂ TIẾT KIỆM TÍN DỤNG COMPUTE UNITS
# ==============================================================================
import time

AUTO_DISCONNECT = True
PRE_DISCONNECT_WAIT_SEC = 10

if AUTO_DISCONNECT:
    if not globals().get('UPLOAD_COMPLETED', False):
        print("⚠️ Upload R2/GCS chưa xác minh thành công — giữ nguyên runtime để tránh mất kết quả!")
    else:
        print(f"⏳ Chờ {PRE_DISCONNECT_WAIT_SEC}s để đồng bộ hoàn toàn trước khi ngắt kết nối...")
        time.sleep(PRE_DISCONNECT_WAIT_SEC)
        print("🛑 Đang hoàn tất và tự động ngắt kết nối Google Colab Pro...")
        try:
            from google.colab import runtime
            runtime.unassign()
            print("✅ Đã ngắt kết nối runtime an toàn!")
        except Exception as e:
            print(f"⚠️ Không thể ngắt runtime tự động: {e}")
else:
    print("ℹ️ AUTO_DISCONNECT đang tắt. Runtime vẫn giữ kết nối để anh tiếp tục làm việc.")